# Curadoria do Dataset CelebA usando CLIP ViT-B/32
Seleciona o maior cluster de imagens com base em similaridade semântica, reduzindo o dataset para ~50k-80k imagens.

In [2]:
# Instalar o CLIP oficial da OpenAI (necessário apenas uma vez)
!pip install git+https://github.com/openai/CLIP.git -q

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 1.5 MB/s eta 0:00:00


In [3]:
import os
import gc
import numpy as np
import pandas as pd
import torch
import clip
from PIL import Image
from tqdm import tqdm
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans

# Configurações
DATA_DIR = "/kaggle/input/datasets/jessicali9530/celeba-dataset/img_align_celeba/img_align_celeba"
OUTPUT_CSV = "curated_files.csv"
TARGET_CLUSTER_SIZE = 70000  # entre 50k e 80k
BATCH_SIZE = 256             # ajuste conforme a VRAM disponível

print("🔍 Carregando lista de imagens...")
all_files = [f for f in os.listdir(DATA_DIR) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
print(f"✅ Encontradas {len(all_files)} imagens.")

🔍 Carregando lista de imagens...
✅ Encontradas 202599 imagens.


In [4]:
# Carregar modelo CLIP ViT-B/32 uma única vez
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"⚙️ Carregando CLIP ViT-B/32 em {device}...")
model, preprocess = clip.load("ViT-B/32", device=device)
model.eval()
print("✅ Modelo carregado.")

⚙️ Carregando CLIP ViT-B/32 em cuda...


100%|████████████████████████████████████████| 338M/338M [00:03<00:00, 117MiB/s]


✅ Modelo carregado.


In [5]:
# Extrair embeddings (features) de todas as imagens
filenames = []
embeddings = []

print(f"🧠 Extraindo embeddings (batch_size={BATCH_SIZE})...")
with torch.no_grad():
    for i in tqdm(range(0, len(all_files), BATCH_SIZE)):
        batch_files = all_files[i:i+BATCH_SIZE]
        filenames.extend(batch_files)
        
        images = []
        for f in batch_files:
            try:
                img = Image.open(os.path.join(DATA_DIR, f)).convert("RGB")
                images.append(preprocess(img))
            except Exception:
                pass
        
        if not images:
            continue
        
        batch_tensor = torch.stack(images).to(device)
        feats = model.encode_image(batch_tensor)   # shape: (batch, 512)
        embeddings.append(feats.cpu().numpy())
        
        # Libera memória a cada 20 batches
        if i % (20 * BATCH_SIZE) == 0 and i > 0:
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

all_embeddings = np.vstack(embeddings)
print(f"✅ Embeddings extraídos: {all_embeddings.shape}")

🧠 Extraindo embeddings (batch_size=256)...


100%|██████████| 792/792 [36:45<00:00,  2.78s/it]

✅ Embeddings extraídos: (202599, 512)


In [6]:
# Redução de dimensionalidade com PCA: 512 -> 64
print("📉 Aplicando PCA (512 -> 64)...")
pca = PCA(n_components=64, random_state=42)
pca_embeddings = pca.fit_transform(all_embeddings)
print(f"PCA explicou {pca.explained_variance_ratio_.sum():.2%} da variância")

📉 Aplicando PCA (512 -> 64)...
PCA explicou 81.26% da variância


In [7]:
# Clusterização com KMeans (k=20)
print("🎯 Executando KMeans (k=20)...")
kmeans = KMeans(n_clusters=20, random_state=42, n_init='auto')
labels = kmeans.fit_predict(pca_embeddings)

# Identificar o maior cluster
unique, counts = np.unique(labels, return_counts=True)
target_idx = unique[np.argmax(counts)]
print(f"🎯 Maior cluster: ID {target_idx} com {counts[target_idx]} imagens.")

🎯 Executando KMeans (k=20)...
🎯 Maior cluster: ID 16 com 15336 imagens.


In [8]:
# Selecionar imagens do maior cluster (respeitando TARGET_CLUSTER_SIZE)
selected_indices = np.where(labels == target_idx)[0]
if len(selected_indices) > TARGET_CLUSTER_SIZE:
    selected_indices = np.random.choice(selected_indices, TARGET_CLUSTER_SIZE, replace=False)

selected_files = [all_files[i] for i in selected_indices]
print(f"💾 Salvando {len(selected_files)} imagens em {OUTPUT_CSV}...")
pd.DataFrame({"filename": selected_files}).to_csv(OUTPUT_CSV, index=False)
print("✅ Concluído! Lista curada gerada.")

💾 Salvando 15336 imagens em curated_files.csv...
✅ Concluído! Lista curada gerada.
